# Exploración de datos — JetBlue (B6), BTS On-Time Performance

Fuente: `data/b6_YYYY_MM.csv` (generados por `src/etl_01_descargar.py`). Datos reales del BTS, sin modificar.

Objetivo: conocer el volumen, la calidad (nulos) y el comportamiento de demoras y cancelaciones antes de cargar el data mart.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"  # carga plotly.js por CDN (notebook liviano)

sys.path.insert(0, str(Path("..") / "src"))
import reglas  # mismas reglas que el ETL (estado del vuelo, rango de demora)

pd.set_option("display.float_format", "{:,.2f}".format)

archivos = sorted(Path("../data").glob("b6_*.csv"))
df = pd.concat([pd.read_csv(a, dtype=str) for a in archivos], ignore_index=True)

NUMERICAS = ["DepDelay", "ArrDelay", "ArrDel15", "TaxiOut", "Cancelled", "Diverted", "Distance",
             "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
for col in NUMERICAS:
    df[col] = pd.to_numeric(df[col])
df["mes"] = df["FlightDate"].str[:7]

print(f"{len(archivos)} archivos, {len(df):,} vuelos, {df.shape[1] - 1} columnas BTS")
print(f"Desde {df['FlightDate'].min()} hasta {df['FlightDate'].max()}")

12 archivos, 240,615 vuelos, 26 columnas BTS
Desde 2025-09-01 hasta 2026-08-31


## 1. Filas por mes

In [2]:
filas_mes = df.groupby("mes").size().rename("vuelos").to_frame()
filas_mes["vuelos_por_dia"] = filas_mes["vuelos"] / df.groupby("mes")["FlightDate"].nunique()
filas_mes

,vuelos,vuelos_por_dia
mes,,
2025-09,16215,540.50
2025-10,18401,593.58
2025-11,19818,660.60
2025-12,20560,663.23
2026-01,18109,584.16
2026-02,17871,638.25
2026-03,21340,688.39
2026-04,20650,688.33
2026-05,19806,638.90


## 2. Porcentaje de nulos por columna

In [3]:
nulos = (df.drop(columns="mes").isna().mean() * 100).sort_values(ascending=False).rename("% nulos")
nulos.to_frame()

,% nulos
CancellationCode,96.44
LateAircraftDelay,72.39
SecurityDelay,72.39
NASDelay,72.39
WeatherDelay,72.39
CarrierDelay,72.39
ArrDelay,3.91
ArrDel15,3.91
TaxiOut,3.56
DepDelay,3.46


Las columnas de causa (`CarrierDelay` … `LateAircraftDelay`) están vacías en la mayoría de filas porque BTS solo las informa en vuelos demorados (ver sección 5). Los nulos de `DepDelay`, `ArrDelay` y `TaxiOut` corresponden a vuelos cancelados o desviados. **No se imputan con 0.**

## 3. Distribución de la demora de llegada (`ArrDelay`)

In [4]:
df["ArrDelay"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("ArrDelay (min)")

,ArrDelay (min)
count,"231,217.00"
mean,14.03
std,64.37
min,-86.00
5%,-30.00
25%,-16.00
50%,-4.00
75%,20.00
90%,72.00
95%,119.00


In [5]:
# Rangos de DimRangoDemora, calculados con reglas.py
nombres_rango = {0: "No aplica (cancelado/desviado)", 1: "Llegó antes de hora", 2: "A tiempo (0–14 min)",
                 3: "Leve (15–44 min)", 4: "Moderada (45–119 min)", 5: "Grave (120+ min)"}
df["estado"] = [reglas.calcular_estado(c, d, x) for c, d, x in zip(df["Cancelled"], df["Diverted"], df["ArrDel15"])]
df["rango"] = [reglas.calcular_rango(e, a) for e, a in zip(df["estado"], df["ArrDelay"])]
rangos = df["rango"].value_counts().sort_index().rename(index=nombres_rango).to_frame("vuelos")
rangos["%"] = rangos["vuelos"] / rangos["vuelos"].sum() * 100
rangos

,vuelos,%
rango,,
No aplica (cancelado/desviado),9398,3.91
Llegó antes de hora,128333,53.34
A tiempo (0–14 min),36452,15.15
Leve (15–44 min),30394,12.63
Moderada (45–119 min),24540,10.20
Grave (120+ min),11498,4.78


In [6]:
# se agrupa en tramos de 5 min antes de graficar (así el notebook no guarda 190k puntos)
tramos = (df["ArrDelay"][df["ArrDelay"].between(-60, 299)] // 5 * 5).value_counts().sort_index()
fig = px.bar(x=tramos.index, y=tramos.values, title="Demora de llegada (entre -60 y 300 min, tramos de 5 min)",
             labels={"x": "Minutos de demora (negativo = llegó antes)", "y": "Vuelos"})
fig.add_vline(x=15, line_dash="dash", annotation_text="15 min (umbral de demora)")
fig.show()

## 4. Vuelos cancelados y desviados

In [7]:
cancel = df.groupby("mes").agg(vuelos=("FlightDate", "size"),
                               cancelados=("Cancelled", "sum"), desviados=("Diverted", "sum"))
cancel.loc["TOTAL"] = cancel.sum()
cancel.astype(int)

,vuelos,cancelados,desviados
mes,,,
2025-09,16215,94,42
2025-10,18401,193,64
2025-11,19818,587,54
2025-12,20560,572,41
2026-01,18109,1201,33
2026-02,17871,1377,37
2026-03,21340,432,83
2026-04,20650,21,57
2026-05,19806,42,49


In [8]:
codigos = {"A": "Aerolínea", "B": "Clima", "C": "Control de tráfico aéreo", "D": "Seguridad"}
causas_cancel = df["CancellationCode"].map(codigos).value_counts().to_frame("cancelados")
causas_cancel["%"] = causas_cancel["cancelados"] / causas_cancel["cancelados"].sum() * 100
causas_cancel

,cancelados,%
CancellationCode,,
Clima,3582,41.76
Control de tráfico aéreo,2909,33.92
Aerolínea,2086,24.32


## 5. ¿Las columnas de causa solo existen cuando `ArrDel15 = 1`?

In [9]:
CAUSAS = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
df["tiene_causas"] = df[CAUSAS].notna().any(axis=1)
pd.crosstab(df["ArrDel15"].fillna(-1).map({1: "Demorado (1)", 0: "No demorado (0)", -1: "NULL (cancelado/desviado)"}),
            df["tiene_causas"].map({True: "con causas", False: "sin causas"}), margins=True)

tiene_causas,con causas,sin causas,All
ArrDel15,,,
Demorado (1),66432,0,66432
NULL (cancelado/desviado),0,9398,9398
No demorado (0),0,164785,164785
All,66432,174183,240615


In [10]:
# En los demorados, la suma de las 5 causas debería igualar ArrDelay
demorados = df[df["ArrDel15"] == 1]
diferencia = (demorados[CAUSAS].sum(axis=1) - demorados["ArrDelay"]).abs()
print(f"Demorados: {len(demorados):,}")
print(f"Suma de causas = ArrDelay en {(diferencia < 1).mean() * 100:.2f}% de los demorados")
print(f"Demorados sin ninguna causa informada (→ 'Sin dato'): {(~demorados['tiene_causas']).sum():,}")

Demorados: 66,432
Suma de causas = ArrDelay en 100.00% de los demorados
Demorados sin ninguna causa informada (→ 'Sin dato'): 0


## 6. % de puntualidad y % de cancelación por mes

Mismas fórmulas que los KPI del data mart, con la definición del *Air Travel Consumer Report* del DOT: puntualidad = a tiempo / vuelos programados (cancelados y desviados cuentan como no puntuales); cancelación = cancelados / vuelos programados. Siempre suma / suma.

In [11]:
kpi = df.groupby("mes").agg(vuelos=("estado", "size"),
                            a_tiempo=("estado", lambda s: (s == reglas.A_TIEMPO).sum()),
                            cancelados=("estado", lambda s: (s == reglas.CANCELADO).sum()))
kpi["% puntualidad"] = kpi["a_tiempo"] / kpi["vuelos"] * 100
kpi["% cancelación"] = kpi["cancelados"] / kpi["vuelos"] * 100
total = kpi[["vuelos", "a_tiempo", "cancelados"]].sum()
kpi.loc["TOTAL"] = [*total, total["a_tiempo"] / total["vuelos"] * 100,
                    total["cancelados"] / total["vuelos"] * 100]
kpi

,vuelos,a_tiempo,cancelados,% puntualidad,% cancelación
mes,,,,,
2025-09,16215,12732,94,78.52,0.58
2025-10,18401,13527,193,73.51,1.05
2025-11,19818,14400,587,72.66,2.96
2025-12,20560,12737,572,61.95,2.78
2026-01,18109,11536,1201,63.70,6.63
2026-02,17871,11142,1377,62.35,7.71
2026-03,21340,14578,432,68.31,2.02
2026-04,20650,15607,21,75.58,0.10
2026-05,19806,16044,42,81.01,0.21


In [12]:
fig = px.line(kpi.drop("TOTAL").reset_index(), x="mes", y=["% puntualidad", "% cancelación"], markers=True,
              title="Puntualidad y cancelación de JetBlue por mes", labels={"value": "%", "variable": ""})
fig.show()

## 7. Participación de cada causa en los minutos de demora

In [13]:
minutos = demorados[CAUSAS].sum().rename({"CarrierDelay": "Aerolínea", "WeatherDelay": "Clima",
                                          "NASDelay": "Control de tráfico aéreo", "SecurityDelay": "Seguridad",
                                          "LateAircraftDelay": "Avión anterior retrasado"})
participacion = (minutos / minutos.sum() * 100).sort_values(ascending=False).to_frame("% de minutos")
participacion

,% de minutos
Avión anterior retrasado,44.93
Aerolínea,27.39
Control de tráfico aéreo,25.00
Clima,2.51
Seguridad,0.18
